# Kiểm thử Kịch bản 1 — Trích xuất kỹ năng NER từ JD và CV

**Mục đích:** Chạy pipeline crawl + clean để lấy JD mới, trích xuất kỹ năng tự động,
sau đó tạo template ground truth để nhóm gán nhãn thủ công.

**Bước thực hiện:**
1. **Phần A:** Crawl + clean tuyển dụng → lấy 20 JD + kỹ năng tự động trích xuất từ 3 nguồn (trừ VietnamWorks)
2. **Phần B:** Trích xuất kỹ năng từ 20 CV trong thư mục `cvs/`
3. **Phần C:** Tạo file ground truth template để gán nhãn tay

---
## ⚙️ Cấu hình chung

In [13]:
import sys, os, json, shutil, subprocess, re
from pathlib import Path

# ── Đường dẫn ──────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path(".").resolve()           # KiemThu/KiemThu_LLM_Extract/
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent    # JobVisualization_BE/
DB_DIR       = PROJECT_ROOT / "Db"           # Db/
OUTPUT_DIR   = NOTEBOOK_DIR                  # Output cùng thư mục notebook
CV_DIR       = NOTEBOOK_DIR / "cvs"          # Thư mục chứa CV kiểm thử

# Python interpreter từ .venv của Db
VENV_PYTHON = DB_DIR / ".venv" / "Scripts" / "python.exe"
PYTHON_EXE  = str(VENV_PYTHON) if VENV_PYTHON.exists() else sys.executable

# Tự tạo thư mục cvs nếu chưa có
CV_DIR.mkdir(exist_ok=True)

# Thêm project root vào sys.path
for p in [str(PROJECT_ROOT), str(PROJECT_ROOT / "Db" / "pipeline" / "clean" / "2_clean_data")]:
    if p not in sys.path:
        sys.path.insert(0, p)

# ── Load .env ───────────────────────────────────────────────────────────────
from dotenv import load_dotenv
load_dotenv(DB_DIR / ".env")

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DB_DIR       : {DB_DIR}")
print(f"OUTPUT_DIR   : {OUTPUT_DIR}")
print(f"CV_DIR       : {CV_DIR}")
print(f"PYTHON_EXE   : {PYTHON_EXE}")

PROJECT_ROOT : F:\HCMUS_KH\LuanVan\JobVisualization_BE
DB_DIR       : F:\HCMUS_KH\LuanVan\JobVisualization_BE\Db
OUTPUT_DIR   : F:\HCMUS_KH\LuanVan\JobVisualization_BE\KiemThu\KiemThu_LLM_Extract
CV_DIR       : F:\HCMUS_KH\LuanVan\JobVisualization_BE\KiemThu\KiemThu_LLM_Extract\cvs
PYTHON_EXE   : F:\HCMUS_KH\LuanVan\JobVisualization_BE\Db\.venv\Scripts\python.exe


---
## 📋 PHẦN A — Crawl + Clean: Lấy 20 JD mới từ 3 nguồn (Không dùng VietnamWorks)

**Cách hoạt động:**
- Ghi đè file từ khóa test `test_keywords_daily.json` thành dạng cấu trúc tier chứa duy nhất `software engineer` để ép pipeline chỉ crawl từ khóa này.
- Gọi `run_etl_pipeline.py` bằng `subprocess` với các env override:
  - 3 nguồn: ITviec + CareerViet + LinkedIn (7 JD/nguồn ≈ 20 JD). **Đã tắt VietnamWorks** do hay bị lẫn tin phi-IT.
  - Tắt bước import vào DB (`PIPELINE_IMPORT=false`)
- Đọc `extracted.json` sinh ra → rút gọn thành `{url, skills}`

In [14]:
# ── A1: Cấu hình env override cho pipeline ──────────────────────────────────

# 3 nguồn × 7 JD/nguồn ≈ 21 JD
JOBS_PER_SOURCE = 10

# Từ khóa muốn chạy crawl
KEYWORDS_TO_CRAWL = ["software engineer"]

# Ghi đè file test_keywords_daily.json theo đúng cấu trúc tier mà hệ thống hỗ trợ
test_keywords_path = DB_DIR / "input" / "test_keywords_daily.json"

# Lưu lại nội dung cũ của file test_keywords nếu có để khôi phục sau khi chạy
old_keywords_content = None
if test_keywords_path.exists():
    try:
        old_keywords_content = test_keywords_path.read_text(encoding="utf-8")
    except Exception:
        pass

# Cấu trúc tier chuẩn để hàm _flatten_keywords_daily phân tích thành công
structured_keywords = {
    "tier_1": KEYWORDS_TO_CRAWL
}

with open(test_keywords_path, "w", encoding="utf-8") as f:
    json.dump(structured_keywords, f, ensure_ascii=False, indent=2)
print(f"✓ Đã cấu trúc lại file test keyword tại: {test_keywords_path.name} thành {structured_keywords}")

# LinkedIn cookie — lấy từ .env (đã cấu hình sẵn)
linkedin_cookie = os.getenv("LINKEDIN_COOKIE", "")
if not linkedin_cookie or "YOUR_LI_AT_HERE" in linkedin_cookie:
    print("⚠️  LINKEDIN_COOKIE chưa được cấu hình trong Db/.env")
    print("   LinkedIn sẽ trả về 0 kết quả. Các nguồn khác vẫn chạy bình thường.")
else:
    print(f"✓ LINKEDIN_COOKIE đã cấu hình: {linkedin_cookie[:30]}...")

PIPELINE_ENV = {
    # ── Chỉ định pipeline sử dụng file test keyword vừa tạo ───────────────────
    "USE_TEST_KEYWORDS":         "true",
    "DAILY_NUM_KEYWORDS":        "1",
    "SELECTED_KEYWORDS":         ", ".join(KEYWORDS_TO_CRAWL),
    "CRAWL_KEYWORDS":            ", ".join(KEYWORDS_TO_CRAWL),
    "RESET_KEYWORD_ROTATION":    "true",   # Reset để chạy từ vị trí đầu tiên
    "ROTATE_EVERY_RUN":          "true",

    # ── 3 nguồn hoạt động tốt nhất, mỗi nguồn JOBS_PER_SOURCE JD ─────────────
    "JOBS_PER_KEYWORD":         str(JOBS_PER_SOURCE),
    "CRAWL_ITVIEC_JOBS":        "1",
    "CRAWL_CAREERVIET_JOBS":    "1",
    "CRAWL_LINKEDIN_JOBS":      "1",
    "CRAWL_VIETNAMWORKS_JOBS":  "0",      # ❌ Loại bỏ VietnamWorks do hay bị lẫn tin phi-IT

    # ── Tắt import vào DB ─────────────────────────────────────────────────────
    "PIPELINE_CRAWL":           "true",
    "PIPELINE_CLEAN":           "true",
    "PIPELINE_IMPORT":          "false",

    # ── Chế độ ngày: tắt lọc để lấy nhiều JD hơn ─────────────────────────────
    "JOB_DATE_MODE":            "off",

    # ── Misc ─────────────────────────────────────────────────────────────────
    "ETL_MAX_THREADS":          "1",
    "ETL_CONFIDENCE_THRESHOLD": "0.0",
    "PYTHONIOENCODING":         "utf-8",
    "PYTHONUNBUFFERED":         "1",
}

print("Cấu hình pipeline:")
for k, v in PIPELINE_ENV.items():
    print(f"   {k} = {v}")

✓ Đã cấu trúc lại file test keyword tại: test_keywords_daily.json thành {'tier_1': ['software engineer']}
⚠️  LINKEDIN_COOKIE chưa được cấu hình trong Db/.env
   LinkedIn sẽ trả về 0 kết quả. Các nguồn khác vẫn chạy bình thường.
Cấu hình pipeline:
   USE_TEST_KEYWORDS = true
   DAILY_NUM_KEYWORDS = 1
   SELECTED_KEYWORDS = software engineer
   CRAWL_KEYWORDS = software engineer
   RESET_KEYWORD_ROTATION = true
   ROTATE_EVERY_RUN = true
   JOBS_PER_KEYWORD = 10
   CRAWL_ITVIEC_JOBS = 1
   CRAWL_CAREERVIET_JOBS = 1
   CRAWL_LINKEDIN_JOBS = 1
   CRAWL_VIETNAMWORKS_JOBS = 0
   PIPELINE_CRAWL = true
   PIPELINE_CLEAN = true
   PIPELINE_IMPORT = false
   JOB_DATE_MODE = off
   ETL_MAX_THREADS = 1
   ETL_CONFIDENCE_THRESHOLD = 0.0
   PYTHONIOENCODING = utf-8
   PYTHONUNBUFFERED = 1


In [15]:
# ── A2: Chạy pipeline crawl + clean ────────────────────────────────────────

print('======================================================================')
print('🎯 DANH SÁCH KEYWORDS SẼ CRAWL:')
print(f'   1. {KEYWORDS_TO_CRAWL[0]}')
print('======================================================================')
print()

run_env = os.environ.copy()
run_env.update(PIPELINE_ENV)

# Tạo file rỗng tạm thời để đánh lừa và bỏ qua bước chuẩn hóa (Normalize)
mock_normalized_path = DB_DIR / "input" / "mock_normalized.json"
with open(mock_normalized_path, "w", encoding="utf-8") as f:
    json.dump([], f)

pipeline_script = DB_DIR / "run_etl_pipeline.py"
cmd = [
    PYTHON_EXE, "-W", "ignore",
    str(pipeline_script),
    "--crawl-mode", "daily",
    "--parallel-crawl",
    "--normalized", str(mock_normalized_path), # Ép pipeline bỏ qua bước Normalize
]

print('🚀 Chạy: run_etl_pipeline.py --crawl-mode daily --parallel-crawl --normalized mock_normalized.json')
print(f'   Nguồn: ITviec + CareerViet + LinkedIn (Bỏ VietnamWorks)')
print(f'   Số lượng: {JOBS_PER_SOURCE} JD/nguồn')
print('(Quá trình mất 3–7 phút, output hiển thị theo thời gian thực)')
print()
print('─' * 70)

try:
    process = subprocess.Popen(
        cmd,
        cwd=str(DB_DIR),
        env=run_env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
    )

    for line in process.stdout:
        print(line, end="", flush=True)

    returncode = process.wait()
finally:
    # Dọn dẹp file mock normalized
    if mock_normalized_path.exists():
        try:
            mock_normalized_path.unlink()
        except Exception:
            pass

    # Khôi phục lại nội dung cũ của file test_keywords_daily.json
    if old_keywords_content is not None:
        try:
            test_keywords_path.write_text(old_keywords_content, encoding="utf-8")
            print()
            print('🔄 Đã khôi phục file test_keywords_daily.json gốc')
        except Exception:
            pass
    else:
        if test_keywords_path.exists():
            try:
                test_keywords_path.unlink()
                print()
                print('🧹 Đã dọn dẹp file test_keywords_daily.json tạm')
            except Exception:
                pass

print('─' * 70)

if returncode == 0:
    print()
    print('✅ Pipeline hoàn tất thành công!')
else:
    print()
    print(f'❌ Pipeline kết thúc với mã lỗi: {returncode}')
    print('   Kiểm tra output bên trên để tìm nguyên nhân.')

🎯 DANH SÁCH KEYWORDS SẼ CRAWL:
   1. software engineer

🚀 Chạy: run_etl_pipeline.py --crawl-mode daily --parallel-crawl --normalized mock_normalized.json
   Nguồn: ITviec + CareerViet + LinkedIn (Bỏ VietnamWorks)
   Số lượng: 10 JD/nguồn
(Quá trình mất 3–7 phút, output hiển thị theo thời gian thực)

──────────────────────────────────────────────────────────────────────
✓ Loaded .env from: F:\HCMUS_KH\LuanVan\JobVisualization_BE\Db\.env
✓ Using .venv Python: F:\HCMUS_KH\LuanVan\JobVisualization_BE\Db\.venv\Scripts\python.exe

✅ Config loaded from input package

📊 KEYWORD & JOB CONFIGURATION

📋 KEYWORD SELECTION:
Tier       Num to Crawl    Method       Enabled   
--------------------------------------------------------------------------------
tier1      1               random       ✓         
tier2      0               sequential   ✗         
tier3      0               sequential   ✗         
--------------------------------------------------------------------------------
TOTAL      1   

In [16]:
# ── A3: Tìm extracted.json mới nhất sinh ra bởi pipeline ───────────────────
data_dir = DB_DIR / "data"

crawl_dirs = sorted(
    [d for d in data_dir.iterdir() if d.is_dir() and re.match(r"crawl_\d{8}_\d{6}", d.name)],
    reverse=True
)

extracted_file = None
for crawl_dir in crawl_dirs:
    candidate = crawl_dir / "clean" / "extracted.json"
    if candidate.exists():
        extracted_file = candidate
        break

if not extracted_file:
    raise FileNotFoundError(
        f"❌ Không tìm thấy extracted.json trong {data_dir}\n"
        "   Pipeline có thể đã thất bại ở bước LLM Extract."
    )

print(f"✓ extracted.json: {extracted_file}")
print(f"  Kích thước     : {extracted_file.stat().st_size / 1024:.1f} KB")

✓ extracted.json: F:\HCMUS_KH\LuanVan\JobVisualization_BE\Db\data\crawl_20260714_184028\clean\extracted.json
  Kích thước     : 427.1 KB


In [17]:
# ── A4: Đọc extracted.json → rút gọn thành {url, skills} ───────────────────
NUM_JDS = 20

with open(extracted_file, encoding="utf-8") as f:
    raw_data = json.load(f)

if not isinstance(raw_data, list):
    raw_data = [raw_data]

print(f"✓ Tổng bản ghi trong extracted.json: {len(raw_data)}")

pipeline_jds_output = []
jds_raw_lines       = []
added_count         = 0

for item in raw_data:
    if added_count >= NUM_JDS:
        break

    # ── Lọc bỏ job không liên quan đến IT (nếu is_it_job xác định là False) ──
    is_it_job = item.get("is_it_job")
    if isinstance(is_it_job, dict):
        is_it_val = is_it_job.get("value")
    else:
        is_it_val = is_it_job
    
    if is_it_val is False:
        continue  # Bỏ qua các job không liên quan đến IT

    url    = item.get("job_url") or item.get("job_posting_url") or ""
    source = item.get("source_name") or ""
    desc   = item.get("description_html") or item.get("requirements_text") or ""
    
    # 🛠️ SỬA LỖI CRASH: Xử lý title là chuỗi (str) hoặc từ điển (dict)
    title_raw = item.get("title") or ""
    if isinstance(title_raw, dict):
        title = title_raw.get("value") or ""
    else:
        title = str(title_raw)

    # Rút gọn skill: chỉ lấy skill_name
    raw_skills = item.get("extracted_skills", [])
    skills = []
    for s in raw_skills:
        if isinstance(s, dict):
            name = s.get("skill_name") or s.get("skill") or ""
            if name:
                skills.append(name)
        elif isinstance(s, str) and s:
            skills.append(s)

    pipeline_jds_output.append({"url": url, "skills": skills})
    added_count += 1
    
    jds_raw_lines.append(
        f"{'='*80}\n"
        f"JD {added_count:02d} | Nguồn: {source} | Keyword: {item.get('search_keyword','')}\n"
        f"Tiêu đề : {title}\n"
        f"URL     : {url}\n"
        f"{'─'*80}\n"
        f"{desc[:3000]}\n\n"
    )
    print(f"  [{added_count:02d}] [{source:<12}] {title[:45]:<45} → {len(skills)} skill")

print()
print(f"✅ Đã lọc và xử lý thành công {added_count} JD công nghệ thông tin.")

✓ Tổng bản ghi trong extracted.json: 24
  [01] [linkedin    ] Software Engineer                             → 35 skill
  [02] [linkedin    ] Software Engineer (Node.js, AWS, Winforms)    → 36 skill
  [03] [linkedin    ] Software Engineer II                          → 11 skill
  [04] [linkedin    ] Software Engineer (Python)                    → 10 skill
  [05] [linkedin    ] Software Engineering                          → 7 skill
  [06] [linkedin    ] System Software Engineer, AI Data Platform    → 20 skill
  [07] [linkedin    ] Solution Engineer                             → 11 skill
  [08] [itviec      ] Senior Embedded Software Engineer (C/C++/AUTO → 32 skill
  [09] [linkedin    ] Software Engineer                             → 34 skill
  [10] [itviec      ] Middle/Senior Software Engineer (Java, Python → 43 skill
  [11] [itviec      ] Software Engineer II (Java/ Golang/ PHP/ Pyth → 15 skill
  [12] [itviec      ] AI-Augmented Software Engineer (.NET & AI Age → 65 skill
  [13] [itvie

In [18]:
# ── A5: Lưu output JD ──────────────────────────────────────────────────────
f_pipeline_jds = OUTPUT_DIR / "pipeline_jds_output.json"
f_jds_raw      = OUTPUT_DIR / "jds_raw_text.txt"

f_pipeline_jds.write_text(
    json.dumps(pipeline_jds_output, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
f_jds_raw.write_text("".join(jds_raw_lines), encoding="utf-8")

print(f"✓ pipeline_jds_output.json — {len(pipeline_jds_output)} JD")
print(f"✓ jds_raw_text.txt        — văn bản gốc để đọc khi gán nhãn")

✓ pipeline_jds_output.json — 20 JD
✓ jds_raw_text.txt        — văn bản gốc để đọc khi gán nhãn


---
## 📋 PHẦN B — Trích xuất kỹ năng từ 20 CV

**Cách hoạt động:** Gọi hàm `extract_student_skills_gemini` từ module `matching_cv`.  
CV được lấy từ thư mục **`KiemThu_LLM_Extract/cvs/`**.

> ⚠️ Đặt file CV (`.pdf` / `.jpg` / `.png`) vào thư mục `cvs/` trước khi chạy phần này.

In [19]:
# ── B1: Quét danh sách CV trong cvs/ ───────────────────────────────────────
NUM_CVS = 20
ALLOWED_EXT = {".pdf", ".jpg", ".jpeg", ".png"}

cv_files = sorted([
    f for f in CV_DIR.rglob("*")
    if f.is_file() and f.suffix.lower() in ALLOWED_EXT
])[:NUM_CVS]

if not cv_files:
    raise FileNotFoundError(
        f"❌ Không tìm thấy file CV trong {CV_DIR}\n"
        f"   Đặt file .pdf/.jpg/.png vào thư mục cvs/ rồi chạy lại."
    )

print(f"✓ Tìm thấy {len(cv_files)} CV:")
for i, f in enumerate(cv_files, 1):
    print(f"  [{i:02d}] {f.name}")

✓ Tìm thấy 20 CV:
  [01] 22127046_BaCong_CV.pdf
  [02] _CV_job__Quang_Huy_Tran.pdf
  [03] CareerNovaCV_HienLuong.pdf
  [04] CV_22127470 _LeHoangYen.pdf
  [05] CV_AI.png
  [06] CV_AI_18.png
  [07] CV_AI_20.jpg
  [08] CV_BA_3.png
  [09] CV_BA_38.png
  [10] CV_BD_25.png
  [11] CV_BE_22.jpg
  [12] CV_BE_24.png
  [13] CV_BE_26.png
  [14] CV_BE_Developer_5.png
  [15] CV_BE_intern_7.jpg
  [16] CV_Business_Analyst.pdf
  [17] CV_Business_Analyst_8.jpg
  [18] CV_DA_28.jpg
  [19] CV_DA_29.jpg
  [20] CV_DA_30.png


In [20]:
# ── B2: Import module matching_cv ───────────────────────────────────────────
from matching_cv.utils import extract_cv_text
from matching_cv.match_cv import extract_student_skills_gemini

print("✓ Import module matching_cv thành công")

✓ Import module matching_cv thành công


In [21]:
# ── B3: Trích xuất kỹ năng từng CV ─────────────────────────────────────────
pipeline_cvs_output = []

for idx, cv_path in enumerate(cv_files, 1):
    filename = cv_path.name
    print(f"[{idx:02d}/{len(cv_files)}] {filename}")

    try:
        cv_text = extract_cv_text(str(cv_path))

        if not cv_text or len(cv_text.strip()) < 50:
            print(f"    ⚠️  Văn bản quá ngắn ({len(cv_text)} ký tự)")
            pipeline_cvs_output.append({"filename": filename, "skills": []})
            continue

        extracted = extract_student_skills_gemini(cv_text)
        skills = [
            item["skill"] for item in extracted
            if isinstance(item, dict) and item.get("skill")
        ]
    except Exception as e:
        print(f"    ❌ Lỗi: {e}")
        skills = []

    pipeline_cvs_output.append({"filename": filename, "skills": skills})
    print(f"    ↳ {len(skills)} skill: {skills[:4]}{'...' if len(skills) > 4 else ''}")

print()
print(f"✅ Hoàn tất {len(pipeline_cvs_output)} CV")

2026-07-14 18:52:30,898 INFO Calling Gemini (attempt 1/5) using key GEMINI_API_KEY_4 (AIzaSyAa...xdIw)


[01/20] 22127046_BaCong_CV.pdf


KeyboardInterrupt: 

In [ ]:
# ── B4: Lưu output CV ──────────────────────────────────────────────────────
f_pipeline_cvs = OUTPUT_DIR / "pipeline_cvs_output.json"
f_pipeline_cvs.write_text(
    json.dumps(pipeline_cvs_output, ensure_ascii=False, indent=2),
    encoding="utf-8"
)
print(f"✓ pipeline_cvs_output.json — {len(pipeline_cvs_output)} CV")

✓ pipeline_cvs_output.json — 20 CV


---
## 📋 PHẦN C — Tạo Ground Truth Template

Sao chép kết quả pipeline thành file ground truth để nhóm chỉnh sửa thủ công.

> **⚠️ Việc cần làm sau khi chạy cell này:**
> 1. Mở `jds_raw_text.txt` → đọc từng JD gốc
> 2. Mở `ground_truth_jds.json` → **thêm/bớt skill** nếu pipeline trích sai
> 3. Mở CV trong `cvs/` → chỉnh sửa `ground_truth_cvs.json` tương ứng
> 4. Chạy `evaluate_accuracy.ipynb`

In [ ]:
# ── C1: Tạo ground_truth_jds.json ──────────────────────────────────────────
f_gt_jds = OUTPUT_DIR / "ground_truth_jds.json"

if f_gt_jds.exists():
    print(f"⚠️  {f_gt_jds.name} đã tồn tại — KHÔNG ghi đè (tránh mất nhãn tay).")
    print(f"   Xóa file thủ công nếu muốn tạo lại từ đầu.")
else:
    shutil.copy(f_pipeline_jds, f_gt_jds)
    print(f"✓ Tạo {f_gt_jds.name}")
    print(f"  → Mở file và chỉnh sửa danh sách skill cho đúng với văn bản gốc")

# ── C2: Tạo ground_truth_cvs.json ──────────────────────────────────────────
f_gt_cvs = OUTPUT_DIR / "ground_truth_cvs.json"

if f_gt_cvs.exists():
    print()
    print(f"⚠️  {f_gt_cvs.name} đã tồn tại — KHÔNG ghi đè (tránh mất nhãn tay).")
    print(f"   Xóa file thủ công nếu muốn tạo lại từ đầu.")
else:
    shutil.copy(f_pipeline_cvs, f_gt_cvs)
    print()
    print(f"✓ Tạo {f_gt_cvs.name}")
    print(f"  → Mở file và chỉnh sửa danh sách skill cho đúng với CV gốc")

✓ Tạo ground_truth_jds.json
  → Mở file và chỉnh sửa danh sách skill cho đúng với văn bản gốc

✓ Tạo ground_truth_cvs.json
  → Mở file và chỉnh sửa danh sách skill cho đúng với CV gốc


In [ ]:
# ── Tóm tắt ────────────────────────────────────────────────────────────────
print()
print('='*65)
print('  TỔNG KẾT — CÁC FILE ĐÃ TẠO')
print('='*65)
files_info = [
    ("pipeline_jds_output.json", "Kết quả pipeline trích xuất 20 JD"),
    ("pipeline_cvs_output.json", "Kết quả pipeline trích xuất 20 CV"),
    ("jds_raw_text.txt",         "Văn bản JD gốc để đọc khi gán nhãn"),
    ("ground_truth_jds.json",    "⚠️  CẦN CHỈNH SỬA — nhãn chuẩn JD"),
    ("ground_truth_cvs.json",    "⚠️  CẦN CHỈNH SỬA — nhãn chuẩn CV"),
]
for fname, desc in files_info:
    fpath = OUTPUT_DIR / fname
    size  = f"{fpath.stat().st_size / 1024:.1f} KB" if fpath.exists() else "chưa tồn tại"
    print(f"  📄 {fname:<35} {size:>8}  — {desc}")

print()
print("⏭️  BƯỚC TIẾP THEO:")
print("  1. Mở jds_raw_text.txt → đọc từng JD")
print("  2. Chỉnh sửa ground_truth_jds.json (thêm/bớt skill cho đúng)")
print("  3. Mở CV trong cvs/ → chỉnh sửa ground_truth_cvs.json")
print("  4. Chạy notebook evaluate_accuracy.ipynb")


  TỔNG KẾT — CÁC FILE ĐÃ TẠO
  📄 pipeline_jds_output.json             12.3 KB  — Kết quả pipeline trích xuất 20 JD
  📄 pipeline_cvs_output.json              5.3 KB  — Kết quả pipeline trích xuất 20 CV
  📄 jds_raw_text.txt                     65.0 KB  — Văn bản JD gốc để đọc khi gán nhãn
  📄 ground_truth_jds.json                12.3 KB  — ⚠️  CẦN CHỈNH SỬA — nhãn chuẩn JD
  📄 ground_truth_cvs.json                 5.3 KB  — ⚠️  CẦN CHỈNH SỬA — nhãn chuẩn CV

⏭️  BƯỚC TIẾP THEO:
  1. Mở jds_raw_text.txt → đọc từng JD
  2. Chỉnh sửa ground_truth_jds.json (thêm/bớt skill cho đúng)
  3. Mở CV trong cvs/ → chỉnh sửa ground_truth_cvs.json
  4. Chạy notebook evaluate_accuracy.ipynb
